In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 


In [2]:
movie = pd.read_csv('movies.csv')
rating = pd.read_csv('ratings.csv')

In [3]:
print(movie.isnull().sum())
print(rating.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [4]:
movie.drop_duplicates(inplace=True)
rating.drop_duplicates(inplace=True)

In [5]:
movie.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
rating.head()

,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523


In [7]:
rating = rating.drop("timestamp",axis=1)

In [8]:
movie_ratings = rating.merge(
    movie,
    on = "movieId"
)

movie_ratings.head()

,userId,movieId,rating,title,genres
0,1,16,4.0,Casino (1995),Crime|Drama
1,1,24,1.5,Powder (1995),Drama|Sci-Fi
2,1,32,4.0,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,4.0,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,4.0,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [9]:
rating_count = movie_ratings.groupby(
    "title"
)["rating"].count().reset_index()

rating_count.rename(
    columns={"rating":"num_ratings"},
    inplace=True
)

In [10]:
movie_ratings = movie_ratings.merge(
    rating_count,
    on="title"
)

In [11]:
movie_ratings = movie_ratings[
    movie_ratings["num_ratings"] >= 50
]

In [12]:
movie_matrix = movie_ratings.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

In [13]:
movie_matrix.fillna(0, inplace=True)

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
similarity = cosine_similarity(
    movie_matrix.T
)

In [16]:
similarity_df = pd.DataFrame(
    similarity,
    index=movie_matrix.columns,
    columns=movie_matrix.columns
)

In [17]:
def recommend(movie_name):

    if movie_name not in similarity_df.columns:
        return ["Movie not found"]

    similar_movies = similarity_df[
        movie_name
    ].sort_values(
        ascending=False
    )[1:11]

    return similar_movies.index.tolist()

In [18]:
recommend("Toy Story (1995)")

['Star Wars: Episode VI - Return of the Jedi (1983)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Back to the Future (1985)',
 'Star Wars: Episode V - The Empire Strikes Back (1980)',
 'Mission: Impossible (1996)',
 'Willy Wonka & the Chocolate Factory (1971)',
 'Toy Story 2 (1999)',
 'Jurassic Park (1993)']

In [19]:
import pickle

pickle.dump(
    similarity_df,
    open("similarity.pkl","wb")
)

pickle.dump(
    movie,
    open("movies.pkl","wb")
)